# End-to-end PyTorch Geometric training, validation, and testing tutorial for a TopologicPy Regression Dataset

## Expected Folder Contents

- `graphs.csv`  
- `nodes.csv`  
- `edges.csv`  
- `meta.yaml`  

---

## This Script

1. Loads the dataset with the PyG helper class  
2. Builds a graph-level GNN model (graph classification by default)  
3. Trains with a train/val split  
4. Evaluates on validation and test sets  
5. Visualizes learning curves, Correlation Scatter Plots, and prints evaluation metrics  

---

## Notes

- **Requires:** `topologicpy >= 0.9.19`, `torch`, `torch_geometric`, `pandas`, `pyyaml`, `numpy`, `plotly`, `scikit-learn`
- **Example Datasets can found at** `https://github.com/wassimj/topologicpy/tree/main/assets/MachineLearning`

### Installation Example

```bash
pip install torch pandas pyyaml numpy plotly scikit-learn
# then install torch-geometric following their official instructions for your OS/CUDA


In [ ]:
# This cell is not needed if you have pip installed topologicpy
import sys
sys.path.append("C:/Users/sarwj/OneDrive - Cardiff University/Documents/GitHub/topologicpy/src")

## 1. Import the needed libraries and utility function

In [ ]:
from __future__ import annotations
import os
import numpy as np
import pandas as pd
from pathlib import Path
import json
import yaml
from topologicpy.PyG import PyG
from topologicpy.Helper import Helper
from topologicpy.Plotly import Plotly

# def pretty_print_metrics(title: str, metrics: dict) -> None:
#     print("\n" + "=" * 80)
#     print(title)
#     print("=" * 80)
#     for k in sorted(metrics.keys()):
#         v = metrics[k]
#         if isinstance(v, float):
#             print(f"{k:30s}: {v:.6f}")
#         else:
#             print(f"{k:30s}: {v}")
#     print("=" * 80 + "\n")

def pretty_print_metrics(title: str, metrics: dict) -> None:
    print("\n" + "=" * 80)
    print(title)
    print("=" * 80)

    # Convert to DataFrame
    df = pd.DataFrame({
        "metric": list(metrics.keys()),
        "value": list(metrics.values())
    })

    # Sort by metric name (to preserve your original behaviour)
    df = df.sort_values("metric").reset_index(drop=True)

    # Format floats to 6 decimal places
    def _format(v):
        if isinstance(v, float):
            return f"{v:.6f}"
        return v

    df["value"] = df["value"].apply(_format)

    print(df.to_string(index=False))

    print("=" * 80 + "\n")

## 2. Check the TopologicPy Version

In [ ]:
print("The script is compatible with TopologicPy v0.9.18 or newer.")
print(Helper.Version())

## 3. Set your renderer:
* Visual studio code: "vscode"
* Google Colab: "colab"
* Browser: "browser"

In [ ]:
renderer = "vscode"

## 4. Specify the Location of the Dataset

In [ ]:
dataset_dir = Path(r"C:\Users\sarwj\OneDrive - Cardiff University\IAAC\2025-26\S3 - Buildings As Graphs\notebooks\Supporting Files\dataset_regression").resolve()

## 5. Load the CSV Dataset (The Example Has Continous Labels, Task is Graph-level Regression)

In [ ]:
# Optional: read meta.yaml (purely informational)
meta_path = dataset_dir / "meta.yaml"
if meta_path.exists():
    meta = yaml.safe_load(meta_path.read_text(encoding="utf-8"))
    print("Loaded meta.yaml:")
    print(json.dumps(meta, indent=2))
else:
    print("meta.yaml not found (this is fine).")

# This dataset has graphs.csv with a categorical 'label' column -> graph classification.
pyg = PyG.ByCSVPath(
    path=str(dataset_dir),
    level="graph",              # "graph" | "node" | "edge" | "link"
    task="regression",      # "classification" | "regression" | "link_prediction"
    graphLabelType="continuous",
    nodeLabelType="categorical",
    edgeLabelType="categorical",
    holdout_group_by="label",

    # If your headers differ, override here (your attached CSVs match defaults):
    # graphIDHeader="graph_id", graphLabelHeader="label",
    # nodeIDHeader="node_id", nodeLabelHeader="label",
    # edgeSRCHeader="src_id", edgeDSTHeader="dst_id", edgeLabelHeader="label",
)

## Baseline

```

    pyg.SetHyperparameters (
        cv="holdout",
        split=(0.70, 0.15, 0.15),
        random_state=42,
        shuffle=True,
        holdout_group_by="label",
        
        epochs=100,
        batch_size=4,          # or 8 if dataset permits
        lr=0.001,
        weight_decay=1e-4,
        optimizer="adamw",
        gradient_clip_norm=1.0,
        early_stopping=True,
        early_stopping_patience=15,
        use_gpu=True,

        conv="sage",
        hidden_dims=(32,32),
        activation="relu",
        dropout=0.10,
        batch_norm=False,
        residual=False,
        pooling="mean",
    )

```

## 6. Set Hyperparameters

In [ ]:
## Baseline
pyg.SetHyperparameters(
    cv="holdout",
    split=(0.70, 0.15, 0.15),
    random_state=53,
    shuffle=True,
    holdout_group_by="label",

    epochs=70,
    batch_size=4,          # or 8 if dataset permits
    lr=0.001,
    weight_decay=1e-4,
    optimizer="adamw",
    gradient_clip_norm=1.0,
    early_stopping=True,
    early_stopping_patience=15,
    use_gpu=True,

    conv="sage",
    hidden_dims=(32,32),
    activation="relu",
    dropout=0.10,
    batch_norm=False,
    residual=False,
    pooling="mean",
)


# Print a compact summary of the current config and inferred dims/classes
print("PyG config summary:")
print(pyg.Summary())

## 6. Train the Model

In [ ]:
history = pyg.Train()  # returns dict of per-epoch curves (loss + metrics when available)
print(history)

## 7. Plot the Training and Validation Loss Curves

In [ ]:
fig_hist = pyg.PlotHistory()
fig_hist.update_layout({"height": 700, "width":800})
fig_hist.show()

## 8. Validate the Model

In [ ]:
val_metrics = pyg.Validate()
pretty_print_metrics("Validation metrics", val_metrics)


## 9. Test the Model

In [ ]:
test_metrics = pyg.Test()
pretty_print_metrics("Test metrics", test_metrics)

## 10. Plot the Parity Chart (for Regression Only)

### For training portion

In [ ]:
pred = pyg.Predict(split="train")
actual = np.array(pred["y_true"]).reshape(-1)
predicted = np.array(pred["pred"]).reshape(-1)
fig = Plotly.FigureByCorrelation(
    actual,
    predicted,
    title="Correlation between Actual and Predicted Values",
    xTitle="Actual Values",
    yTitle="Predicted Values",
    showIdentity = True,
    showBestFit=True,
    dotSize=6,
    dotColor="blue",
    lineColor="purple",
    width=900,
    height=800,
    theme='default',
    backgroundColor="lightgrey",
)

Plotly.Show(fig)

### For validation portion

In [ ]:
pred = pyg.Predict(split="val")
actual = np.array(pred["y_true"]).reshape(-1)
predicted = np.array(pred["pred"]).reshape(-1)
fig = Plotly.FigureByCorrelation(
    actual,
    predicted,
    title="Correlation between Actual and Predicted Values",
    xTitle="Actual Values",
    yTitle="Predicted Values",
    showIdentity = True,
    showBestFit=True,
    dotSize=6,
    dotColor="blue",
    lineColor="purple",
    width=900,
    height=800,
    theme='default',
    backgroundColor="lightgrey",
)

Plotly.Show(fig)

### For testing portion

In [ ]:
pred = pyg.Predict(split="test")
actual = np.array(pred["y_true"]).reshape(-1)
predicted = np.array(pred["pred"]).reshape(-1)
fig = Plotly.FigureByCorrelation(
    actual,
    predicted,
    title="Correlation between Actual and Predicted Values",
    xTitle="Actual Values",
    yTitle="Predicted Values",
    showIdentity = True,
    showBestFit=True,
    dotSize=6,
    dotColor="blue",
    lineColor="purple",
    width=900,
    height=800,
    theme='default',
    backgroundColor="lightgrey",
)

Plotly.Show(fig)

### For the whole dataset

In [ ]:
pred = pyg.Predict(split="all")
actual = np.array(pred["y_true"]).reshape(-1)
predicted = np.array(pred["pred"]).reshape(-1)
fig = Plotly.FigureByCorrelation(
    actual,
    predicted,
    title="Correlation between Actual and Predicted Values",
    xTitle="Actual Values",
    yTitle="Predicted Values",
    showIdentity = True,
    showBestFit=True,
    dotSize=6,
    dotColor="blue",
    lineColor="purple",
    width=900,
    height=800,
    theme='default',
    backgroundColor="lightgrey",
)

Plotly.Show(fig)